In [1]:
# SpeechBrain 설치
!pip install speechbrain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.8/824.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.8/117.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.2/722.2 kB 30.6 MB/s eta 0:00:00


In [2]:
# Colab에서 최신 버전의 torchaudio와 speechbrain을 설치합니다.
!pip install -U torch torchaudio
!pip install -U speechbrain

In [3]:
!pip install speechbrain torchaudio

In [18]:
## CSV파일 audio_path 컬럼을 코랩드라이브에 맞 경로 수정

import pandas as pd

# 원본 CSV 파일 경로
csv_path = '/content/drive/MyDrive/DAT/NLP/finetuning/whisper_finetuning_data_ds_5.csv'

# 데이터 로드
data_df = pd.read_csv(csv_path)

# 경로 변경: audio_path 열에서 특정 문자열을 다른 문자열로 대체
old_path = "C:/Users/MATH-1/dat_nlp/converted_wav_files"
new_path = "/content/drive/MyDrive/DAT/NLP/음성데이터"
data_df["audio_path"] = data_df["audio_path"].str.replace(old_path, new_path, regex=False)
#data_df["audio_path"] = data_df["audio_path"].str.replace("\\", "/", regex=False)

# 수정된 데이터 확인
print(data_df.head())

# 수정된 데이터 저장
modified_csv_path = '/content/drive/MyDrive/DAT/NLP/finetuning/modified_whisper_finetuning_data_ds_5.csv'
data_df.to_csv(modified_csv_path, index=False)
print(f"수정된 CSV 파일이 저장되었습니다: {modified_csv_path}")


                                          audio_path  \
0  /content/drive/MyDrive/DAT/NLP/음성데이터/f1...   
1  /content/drive/MyDrive/DAT/NLP/음성데이터/f1...   
2  /content/drive/MyDrive/DAT/NLP/음성데이터/f1...   
3  /content/drive/MyDrive/DAT/NLP/음성데이터/f1...   
4  /content/drive/MyDrive/DAT/NLP/음성데이터/f1...   

                                   transcription  start_time  end_time  
0       세로토닌 재흡수를 억제하면 우울이나 불안 symtom을 완화할 수 있어.       2.249    11.020  
1      그럼 한올파모티딘정은 어떤 증상을 완화하는 데 쓰이는 medicine이야?      12.521    22.201  
2  white 색의 원형 필름코팅정인데 위궤양이나 위식도 역류질환을 치료하는데 쓰여.      24.397    36.839  
3  medicine 먹을 때 속 쓰리지 말라고 보통 20밀리그램 단위로 복용할 거야.      39.465    49.412  
4            coronavirus메디슨 어떻해 할거야? 접종할거야? 말거야?      50.539    60.204  
수정된 CSV 파일이 저장되었습니다: /content/drive/MyDrive/DAT/NLP/finetuning/modified_whisper_finetuning_data_ds_5.csv


In [39]:
import pandas as pd
import librosa
from torch.utils.data import Dataset, DataLoader

class PaddedSpeechDataset(Dataset):
    def __init__(self, data, max_length=16000 * 5):  # 5초 기준 (16kHz 샘플링)
        self.data = data
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data.iloc[idx]
        audio_path = item["audio_path"]

        # 오디오 로드 및 패딩
        waveform, _ = librosa.load(audio_path, sr=16000)
        waveform = torch.tensor(waveform)
        if len(waveform) > self.max_length:
            waveform = waveform[:self.max_length]
        else:
            waveform = torch.cat([waveform, torch.zeros(self.max_length - len(waveform))])
        waveform = waveform.unsqueeze(0)  # 채널 차원 추가

        # 텍스트 확인
        transcription = item.get("transcription", None)
        if transcription is None:
            raise ValueError(f"Transcription이 비어 있습니다: {item}")

        transcription = item["transcription"]
        return waveform, transcription


In [40]:
# CSV 파일 로드
csv_path = "/content/drive/MyDrive/DAT/NLP/finetuning/modified_whisper_finetuning_data_ds_5.csv"  # 실제 CSV 파일 경로로 변경
data_df = pd.read_csv(csv_path)

# 데이터셋 및 데이터로더 생성
train_dataset = PaddedSpeechDataset(data_df)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)


In [41]:
# DataLoader 출력 확인
for batch in train_loader:
    try:
        waveforms, transcriptions = batch
        print("Waveforms shape:", waveforms.shape)  # 텐서 크기 확인
        print("Transcriptions:", transcriptions)
        break
    except Exception as e:
        print(f"Error in DataLoader: {e}")


Waveforms shape: torch.Size([4, 1, 80000])
Transcriptions: ('막상 내가 military 가니 그런 프로그램이 사라졌어.', '나 요즘 넷플릭스로 드라마 그레이아나토미 보고 있는데 재미있어.', '너 오늘 Gearlounge에 씬디싸이져 사러 간다고 했지?', '그러냐 그럼 freesia 꽃말이 뭔지도 알아? 내가 방금 봤다는 꽃 이름인데 꽃말은 모르겠네.')


In [43]:
# Null 값 확인
print(data_df.isnull().sum())

audio_path       0
transcription    0
start_time       0
end_time         0
dtype: int64


In [45]:
# 입력 텐서의 크기 확인
print("Waveforms shape:", waveforms.shape)

# 모델이 요구하는 입력 크기로 변환
if waveforms.dim() == 3:  # (batch_size, 1, length)
    waveforms = waveforms.squeeze(1)  # (batch_size, length)

print("Modified Waveforms shape:", waveforms.shape)


Waveforms shape: torch.Size([4, 1, 80000])
Modified Waveforms shape: torch.Size([4, 80000])


In [46]:
# 모델의 구조 확인
print(asr_model.mods)

# 더미 입력값으로 모델 테스트
dummy_input = torch.randn(4, 80000)  # (batch_size, length)
try:
    dummy_output = asr_model.mods.encoder(dummy_input)
    print("Dummy output shape:", dummy_output.shape)
except Exception as e:
    print(f"Error with dummy input: {e}")


ModuleDict(
  (normalizer): InputNormalization()
  (encoder): LengthsCapableSequential(
    (compute_features): Fbank(
      (compute_STFT): STFT()
      (compute_fbanks): Filterbank()
      (compute_deltas): Deltas()
      (context_window): ContextWindow()
    )
    (normalize): InputNormalization()
    (model): CRDNN(
      (CNN): Sequential(
        (block_0): CNN_Block(
          (conv_1): Conv2d(
            (conv): Conv2d(1, 128, kernel_size=(3, 3), stride=(1, 1))
          )
          (norm_1): LayerNorm(
            (norm): LayerNorm((40, 128), eps=1e-05, elementwise_affine=True)
          )
          (act_1): LeakyReLU(negative_slope=0.01)
          (conv_2): Conv2d(
            (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1))
          )
          (norm_2): LayerNorm(
            (norm): LayerNorm((40, 128), eps=1e-05, elementwise_affine=True)
          )
          (act_2): LeakyReLU(negative_slope=0.01)
          (pooling): Pooling1d(
            (pool_layer): Max

In [44]:
# 모델 입력값 디버깅
for batch in train_loader:
    waveforms, transcriptions = batch
    print("Waveforms shape:", waveforms.shape)  # 배치 크기 확인
    print("First waveform:", waveforms[0])      # 첫 번째 텐서 출력
    print("Transcriptions:", transcriptions)    # 전사 데이터 확인

    # 모델 입력 확인
    try:
        encoder_output = asr_model.mods.encoder(waveforms)
        print("Encoder output shape:", encoder_output.shape)
    except Exception as e:
        print(f"Error in model encoder: {e}")
    break


Waveforms shape: torch.Size([4, 1, 80000])
First waveform: tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -6.1035e-05,
         -7.6294e-05, -7.6294e-05]])
Transcriptions: ('Roland의 건반들은 정말 최고의 Polyphonic Syns야.', '그러니까 평소에 Natrium 섭취를 적게 먹으라 했잖아.', '그렇다면 한명만 길게 Stand down을 받아?', '너희 치과에 요즘도 unisem 입고되니? 우리 치과는 요새 안들어와.')
Error in model encoder: 'NoneType' object is not subscriptable


In [34]:
from speechbrain.pretrained import EncoderDecoderASR

# 사전 학습된 SpeechBrain 모델 불러오기
asr_model = EncoderDecoderASR.from_hparams(
    source="speechbrain/asr-crdnn-rnnlm-librispeech",
    savedir="pretrained_model"
)


/usr/local/lib/python3.10/dist-packages/speechbrain/processing/features.py:1311: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  stats = torch.load(path, map_location=device)


In [35]:
import torch.nn as nn

class LanguagePosteriorBias(nn.Module):
    def __init__(self, input_dim, num_languages=2):
        super(LanguagePosteriorBias, self).__init__()
        self.language_layer = nn.Linear(input_dim, num_languages)

    def forward(self, x):
        # x: [batch_size, seq_length, input_dim]
        language_probs = torch.softmax(self.language_layer(x), dim=-1)
        return language_probs


In [36]:
from torch.optim import Adam
from torch.nn import CTCLoss

# 손실 함수 및 옵티마이저
asr_criterion = CTCLoss(blank=0)  # CTC 손실 함수
lpb_criterion = nn.CrossEntropyLoss()  # 언어 후확률 손실 함수
optimizer = Adam(asr_model.mods.parameters(), lr=1e-4)


In [48]:
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch in train_loader:
        waveforms, transcriptions = batch

        # 입력 텐서 크기 확인 및 변환
        if waveforms.dim() == 3:  # (batch_size, 1, length)
            waveforms = waveforms.squeeze(1)  # (batch_size, length)

        # 정규화
        waveforms = (waveforms - waveforms.mean(dim=1, keepdim=True)) / (waveforms.std(dim=1, keepdim=True) + 1e-5)

        # 인코더 출력
        try:
            encoder_output = asr_model.mods.encoder(waveforms)
            if encoder_output is None:
                raise ValueError("Encoder output is None. Check model configuration.")
            print("Encoder output shape:", encoder_output.shape)
        except Exception as e:
            print(f"Error in encoder: {e}")
            break

        # 학습 루프 계속 (CTC 손실 계산 및 업데이트)
        log_probs = torch.log_softmax(encoder_output, dim=-1)
        targets = asr_model.tokenizer(transcriptions)
        input_lengths = torch.full(size=(log_probs.size(0),), fill_value=log_probs.size(1), dtype=torch.long)
        target_lengths = torch.tensor([len(t) for t in targets])

        asr_loss = asr_criterion(log_probs, targets, input_lengths, target_lengths)

        optimizer.zero_grad()
        asr_loss.backward()
        optimizer.step()

        epoch_loss += asr_loss.item()

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss:.4f}")


Error in encoder: 'NoneType' object is not subscriptable
Epoch 1/5, Loss: 0.0000
Error in encoder: 'NoneType' object is not subscriptable
Epoch 2/5, Loss: 0.0000
Error in encoder: 'NoneType' object is not subscriptable
Epoch 3/5, Loss: 0.0000
Error in encoder: 'NoneType' object is not subscriptable
Epoch 4/5, Loss: 0.0000
Error in encoder: 'NoneType' object is not subscriptable
Epoch 5/5, Loss: 0.0000
